# Liconic STX

The Liconic STX line of automated incubators comes in a variety of sizes (STX 44, STX 110, STX 220, STX 280, STX 500, STX 1000 -- corresponding to the number of plate positions) and climate control options:

| Suffix | Climate type | Temp. control | Active cooling | Humidity control |
|---|---|---|---|---|
| IC | Incubator | yes | no | no |
| HC | Humid Cooler | yes | yes | no |
| DC2 | Dry Storage | yes | no | yes |
| HR | Humid Wide Range | yes | yes | no |
| DR2 | Dry Wide Range | yes | no | yes |
| AR | Humidity Controlled | yes | no | yes |
| DF | Deep Freezer | yes | yes | no |
| NC | No Climate | no | no | no |
| DH | Dehumidifier | yes | no | yes |

Cassettes are available for plate heights from 5 mm to 104 mm and can be mixed within a single unit.

Depending on configuration, the STX supports:

- [Automated retrieval](../../capabilities/automated-retrieval) (store/fetch plates by position or strategy)
- [Temperature control](../../capabilities/temperature-control) (heating, active cooling on HC/HR/DF models)
- [Humidity control](../../capabilities/humidity-control) (on DC2/DR2/AR/DH models)
- [Shaking](../../capabilities/shaking) (optional internal shaker)
- [Barcode scanning](../../capabilities/barcode-scanning) (optional internal scanner)

## Setup

Create a `Liconic` instance with the model string (e.g. `"STX220_HC"`), the serial port, and a list of rack cassettes matching your physical configuration. Connect via RS232.

In [ ]:
from pylabrobot.liconic import Liconic
from pylabrobot.liconic.racks import liconic_rack_17mm_22, liconic_rack_44mm_10
from pylabrobot.resources import Coordinate

racks = [
    liconic_rack_44mm_10("cassette_0"),
    liconic_rack_44mm_10("cassette_1"),
    liconic_rack_44mm_10("cassette_2"),
    liconic_rack_17mm_22("cassette_3"),
    liconic_rack_17mm_22("cassette_4"),
    liconic_rack_17mm_22("cassette_5"),
    liconic_rack_17mm_22("cassette_6"),
    liconic_rack_17mm_22("cassette_7"),
    liconic_rack_17mm_22("cassette_8"),
    liconic_rack_17mm_22("cassette_9"),
]

incubator = Liconic(
    name="incubator",
    liconic_model="STX220_HC",
    port="/dev/ttyUSB0",  # replace with your port
    racks=racks,
    loading_tray_location=Coordinate(x=0, y=0, z=0),
)

await incubator.setup()

Racks can be mixed -- here we use 44 mm cassettes (10 plates each) for taller plates and 17 mm cassettes (22 plates each) for standard-height plates. See `pylabrobot.liconic.racks` for the full list of available cassette types.

## Storing and retrieving plates

Place a plate on the loading tray, then call `take_in_plate` to store it. You can specify a strategy (`"smallest"` picks the smallest free site that fits, `"random"` picks a random one) or pass a specific rack site. For the full API, see [Automated Retrieval](../../capabilities/automated-retrieval).

In [ ]:
from pylabrobot.resources import Azenta4titudeFrameStar_96_wellplate_200ul_Vb

plate = Azenta4titudeFrameStar_96_wellplate_200ul_Vb(name="my_plate")
incubator.loading_tray.assign_child_resource(plate)

await incubator.take_in_plate("smallest")  # store in the smallest free site that fits

Retrieve a plate by name:

In [ ]:
await incubator.fetch_plate_to_loading_tray(plate_name="my_plate")
retrieved = incubator.loading_tray.resource

You can also store at a specific rack site or use `"random"`:

In [ ]:
await incubator.take_in_plate("random")          # random free site
# await incubator.take_in_plate(racks[3].sites[0])  # specific rack and position

Print a summary of what is stored where:

In [ ]:
print(incubator.summary())

## Temperature control

Models with temperature control (all except `_NC`) expose a {class}`~pylabrobot.capabilities.temperature_controlling.temperature_controller.TemperatureController` on `incubator.tc`. For the full API, see [Temperature Control](../../capabilities/temperature-control).

In [ ]:
await incubator.tc.set_temperature(37.0)

In [ ]:
current = await incubator.tc.request_temperature()
print(f"{current:.1f} °C")

## Humidity control

Models with humidity control (`_DC2`, `_DR2`, `_AR`, `_DH`) expose a {class}`~pylabrobot.capabilities.humidity_controlling.humidity_controller.HumidityController` on `incubator.humidity_controller`. For the full API, see [Humidity Control](../../capabilities/humidity-control).

Humidity is expressed as a fraction (0.0--1.0), not a percentage.

In [ ]:
# Only available on models with humidity control (DC2, DR2, AR, DH)
# await incubator.humidity_controller.set_humidity(0.95)  # 95% RH
# current_rh = await incubator.humidity_controller.request_humidity()

## Shaking

If your STX has an internal shaker installed, pass `has_shaker=True` when constructing the `Liconic`. The shaker is then available at `incubator.shaker`. For the full API, see [Shaking](../../capabilities/shaking).

Note: shaking speed on the Liconic is specified in Hz (1.0--50.0), not RPM.

In [ ]:
# Only if has_shaker=True was passed during construction:
# await incubator.shaker.shake(speed=10.0)   # 10 Hz
# await incubator.shaker.stop_shaking()

## CO2 and N2 gas control

The backend exposes direct methods for CO2 and N2 gas level control (when the hardware is equipped):

In [ ]:
# CO2 control (if installed)
# await incubator.driver.set_co2_level(0.05)  # 5% CO2
# co2 = await incubator.driver.request_co2_level()

# N2 control (if installed)
# await incubator.driver.set_n2_level(0.10)  # 10% N2
# n2 = await incubator.driver.request_n2_level()

## Teardown

In [ ]:
await incubator.stop()